In [33]:
import yaml
import os
import polars as pl
from datetime import datetime ,date

with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

data_dir = config["data_dir"]
print(data_dir  )
if os.path.exists(data_dir) and os.path.isdir(data_dir):
    list_of_files = os.listdir(data_dir)
else:
    list_of_files = []
print(list_of_files)

E:\smallcaps_data
['AIXI.parquet', 'APUS.parquet', 'BAOS.parquet', 'BEAT.parquet', 'BIAF.parquet', 'CAN.parquet', 'CDTG.parquet', 'CHAI.parquet', 'CHPT.parquet', 'CISS.parquet', 'COOT.parquet', 'CYAB.parquet', 'FBGL.parquet', 'FLYE.parquet', 'GCDT.parquet', 'GIPR.parquet', 'GRI.parquet', 'GRML.parquet', 'IMCC.parquet', 'IMRN.parquet', 'LIDR.parquet', 'logs', 'MEDS.parquet', 'NCPL.parquet', 'NCRA.parquet', 'NEOV.parquet', 'OFAL.parquet', 'OLB.parquet', 'OLOX.parquet', 'output', 'PDSB.parquet', 'PFSA.parquet', 'SBFM.parquet', 'SDEV.parquet', 'SKYQ.parquet', 'SOAR.parquet', 'SPHL.parquet', 'STAK.parquet', 'TNMG.parquet', 'TOPP.parquet', 'TRUG.parquet', 'USDE.parquet', 'VEEA.parquet', 'VRA.parquet', 'VRME.parquet', 'XHLD.parquet', 'XPON.parquet', 'XTNT.parquet', '_catalog.parquet']


In [34]:
# list_of_files = [f for f in list_of_files if f.endswith(".parquet")]
list_of_tickers = [f for f in list_of_files if not f.startswith("_catalog") and f.endswith(".parquet")]
catalog = [f for f in list_of_files if f.startswith("_catalog") and f.endswith(".parquet")][0]
# parquet_path = os.path.join(data_dir, f"{ticker}.parquet")
catalog_path = os.path.join(data_dir, f"{catalog}")

df_catalog = pl.read_parquet(catalog_path)['ticker','end'].sort('ticker')
df_catalog = df_catalog.with_columns(
    pl.col('end').dt.date().alias('date')
)
df_catalog


ticker,end,date
str,"datetime[μs, UTC]",date
"""AIXI""",2026-08-07 23:59:00 UTC,2026-08-07
"""AIXI""",2026-08-28 23:59:00 UTC,2026-08-28
"""APUS""",2026-09-04 20:32:00 UTC,2026-09-04
"""APUS""",2026-09-24 23:59:00 UTC,2026-09-24
"""APUS""",2026-10-07 23:59:00 UTC,2026-10-07
…,…,…
"""XHLD""",2026-10-06 23:59:00 UTC,2026-10-06
"""XPON""",2026-08-04 20:05:00 UTC,2026-08-04
"""XPON""",2026-08-24 23:59:00 UTC,2026-08-24


In [35]:
df_backtest_runs = (
    df_catalog
    .select("ticker", "date")
    .unique()
    .sort(["ticker", "date"])
)
df_backtest_runs


ticker,date
str,date
"""AIXI""",2026-08-07
"""AIXI""",2026-08-28
"""APUS""",2026-09-04
"""APUS""",2026-09-24
"""APUS""",2026-10-07
…,…
"""XHLD""",2026-10-06
"""XPON""",2026-08-04
"""XPON""",2026-08-24


In [36]:
tickers_days = {
    row["ticker"]: row["days"]
    for row in (
        df_catalog
        .select(
            "ticker",
            pl.col("end").dt.strftime("%Y-%m-%d").alias("date"),
        )
        .group_by("ticker")
        .agg(
            pl.col("date")
            .unique()
            .sort(descending=True)
            .alias("days")
        )
        .sort("ticker")
        .to_dicts()
    )
}

tickers_days

{'AIXI': ['2026-08-28', '2026-08-07'],
 'APUS': ['2026-10-07', '2026-09-24', '2026-09-04'],
 'BAOS': ['2026-09-04', '2026-08-14'],
 'BEAT': ['2026-10-05', '2026-09-15'],
 'BIAF': ['2026-09-04', '2026-08-14'],
 'CAN': ['2026-08-21', '2026-07-31'],
 'CDTG': ['2026-09-04', '2026-08-14'],
 'CHAI': ['2026-08-28', '2026-08-07'],
 'CHPT': ['2026-09-03', '2026-08-14'],
 'CISS': ['2026-08-28', '2026-08-07'],
 'COOT': ['2026-08-31', '2026-08-11'],
 'CYAB': ['2026-08-28', '2026-08-07'],
 'FBGL': ['2026-09-22', '2026-09-02'],
 'FLYE': ['2026-10-08', '2026-09-18', '2026-09-01', '2026-08-12'],
 'GCDT': ['2026-09-23', '2026-09-03'],
 'GIPR': ['2026-08-25', '2026-08-24', '2026-08-04'],
 'GRI': ['2026-09-04', '2026-08-14'],
 'GRML': ['2026-09-25', '2026-09-04'],
 'IMCC': ['2026-09-22', '2026-09-02'],
 'IMRN': ['2026-09-04', '2026-08-14'],
 'LIDR': ['2026-09-01', '2026-08-12'],
 'MEDS': ['2026-10-08', '2026-09-18'],
 'NCPL': ['2026-09-21', '2026-09-01'],
 'NCRA': ['2026-08-31', '2026-08-11'],
 'NEOV': [

In [42]:
from datetime import date
import os
import polars as pl
DAYS_BEFORE = 1
columns = [
    "datetime",
    "ticker",
    "target_day",
    "open",
    "high",
    "low",
    "close",
    "volume",
]

df_backtest_runs = []

for ticker, target_days in tickers_days.items():
    parquet_path = os.path.join(data_dir, f"{ticker}.parquet")

    if not os.path.isfile(parquet_path):
        raise FileNotFoundError(
            f"No existe el Parquet para {ticker}: {parquet_path}"
        )

    ticker_df = (
        pl.read_parquet(parquet_path)
        .with_columns(
            pl.col("datetime").dt.date().alias("trading_day"),
            pl.lit(ticker).alias("ticker"),
        )
        .sort("datetime")
    )

    available_days = (
        ticker_df
        .select("trading_day")
        .unique()
        .sort("trading_day")
        .get_column("trading_day")
        .to_list()
    )

    for target_day_text in target_days:
        target_day = date.fromisoformat(target_day_text)

        window_days = [
            trading_day
            for trading_day in available_days
            if trading_day <= target_day
        ][-DAYS_BEFORE:]
        # print(window_days)
        if len(window_days) < DAYS_BEFORE:
            # raise ValueError(
            #     f"{ticker} no tiene 10 ruedas disponibles hasta {target_day}"
            # )
            print(f"Se omite {ticker} / {target_day}: no tiene 10 ruedas disponibles")
            continue

        run_df = (
            ticker_df
            .filter(pl.col("trading_day").is_in(window_days))
            .with_columns(pl.lit(target_day).alias("target_day"))
            .select(columns)
            .sort("datetime")
        )
        df_backtest_runs.append(run_df)

In [46]:
df_backtest_runs[0]

datetime,ticker,target_day,open,high,low,close,volume
"datetime[μs, UTC]",str,date,f64,f64,f64,f64,i64
2026-08-28 08:00:00 UTC,"""AIXI""",2026-08-28,0.575,0.575,0.5602,0.572,40312
2026-08-28 08:01:00 UTC,"""AIXI""",2026-08-28,0.5707,0.5829,0.562,0.5685,29729
2026-08-28 08:02:00 UTC,"""AIXI""",2026-08-28,0.5684,0.5684,0.561,0.561,3391
2026-08-28 08:03:00 UTC,"""AIXI""",2026-08-28,0.561,0.561,0.5555,0.561,7291
2026-08-28 08:04:00 UTC,"""AIXI""",2026-08-28,0.5603,0.568,0.5603,0.5604,1422
…,…,…,…,…,…,…,…
2026-08-28 23:52:00 UTC,"""AIXI""",2026-08-28,0.5355,0.5355,0.5355,0.5355,585
2026-08-28 23:54:00 UTC,"""AIXI""",2026-08-28,0.5355,0.5355,0.5355,0.5355,733
2026-08-28 23:55:00 UTC,"""AIXI""",2026-08-28,0.5355,0.5448,0.5355,0.5448,3005


In [49]:
import numpy as np
import polars as pl

def acf1_minute_bars(run_df):
    minute_returns = (
        run_df
        .sort("datetime")
        .with_columns(
            pl.col("datetime").dt.date().alias("trading_day")
        )
        .with_columns(
            pl.col("close")
            .pct_change()
            .over("trading_day")
            .alias("minute_return")
        )
        .with_columns(
            pl.col("minute_return")
            .shift(1)
            .over("trading_day")
            .alias("previous_minute_return")
        )
        .drop_nulls(["minute_return", "previous_minute_return"])
    )

    return acf1(
        minute_returns["previous_minute_return"].to_numpy(),
        minute_returns["minute_return"].to_numpy(),
    )
def acf1(previous_returns, current_returns):
    previous_returns = np.asarray(previous_returns, dtype=float)
    current_returns = np.asarray(current_returns, dtype=float)

    if (
        len(previous_returns) < 2
        or np.std(previous_returns) == 0
        or np.std(current_returns) == 0
    ):
        return np.nan

    return np.corrcoef(previous_returns, current_returns)[0, 1]
kpi_rows = []

for run_df in df_backtest_runs:
    kpi_rows.append(
        {
            "ticker": run_df.get_column("ticker")[0],
            "final_day": run_df.get_column("target_day")[0],
            "acf1": acf1_minute_bars(run_df),
        }
    )

df_kpis = pl.DataFrame(kpi_rows)
df_kpis

# df_kpis = pl.DataFrame(kpi_rows)
# df_kpis

ticker,final_day,acf1
str,date,f64
"""AIXI""",2026-08-28,-0.175962
"""AIXI""",2026-08-07,-0.083814
"""APUS""",2026-10-07,-0.143421
"""APUS""",2026-09-24,0.132292
"""APUS""",2026-09-04,-0.301413
…,…,…
"""XHLD""",2026-09-16,-0.394975
"""XPON""",2026-08-24,-0.079286
"""XPON""",2026-08-04,0.044046


In [50]:
import plotly.graph_objects as go

plot_kpis = df_kpis.sort(["ticker", "final_day"]).with_columns(
    (
        pl.col("ticker")
        + " | "
        + pl.col("final_day").cast(pl.String)
    ).alias("label")
)

acf_values = plot_kpis["acf1"].to_list()

fig = go.Figure(
    go.Bar(
        x=plot_kpis["label"].to_list(),
        y=acf_values,
        marker_color=[
            "#2ca02c" if acf >= 0 else "#d62728"
            for acf in acf_values
        ],
        hovertemplate=(
            "Ticker: %{customdata[0]}<br>"
            "Día final: %{customdata[1]}<br>"
            "ACF(1): %{y:.4f}"
            "<extra></extra>"
        ),
        customdata=list(
            zip(
                plot_kpis["ticker"].to_list(),
                plot_kpis["final_day"].cast(pl.String).to_list(),
            )
        ),
    )
)

fig.add_hline(y=0, line_color="gray", line_width=1)

fig.update_layout(
    title="Autocorrelación de retornos diarios — ACF(1)",
    xaxis_title="Ticker | Día final",
    yaxis_title="ACF(1)",
    xaxis_tickangle=-45,
    height=600,
)

fig.show()